In [ ]:
from google.colab import files
import pandas as pd
import numpy as np

uploaded = files.upload()
uploaded.keys()

Saving geo_sample_labelled (1).csv to geo_sample_labelled (1).csv
Saving star_uuid_patient_map (1) (1).csv to star_uuid_patient_map (1) (1).csv
Saving tcga_sample_labelled (1).csv to tcga_sample_labelled (1).csv


dict_keys(['geo_sample_labelled (1).csv', 'star_uuid_patient_map (1) (1).csv', 'tcga_sample_labelled (1).csv'])

In [ ]:
# Read
tcga = pd.read_csv('tcga_sample_labelled (1).csv')
geo  = pd.read_csv('geo_sample_labelled (1).csv')
star = pd.read_csv('star_uuid_patient_map (1) (1).csv')

# TCGA: ensure canonical sample_id, dataset
# (rename if your sample column is called something else)
if 'sample_id' not in tcga.columns:
    # guess the column; change 'barcode' if needed
    tcga = tcga.rename(columns={'SAMPLE_ID': 'sample_id'})
tcga['dataset'] = 'TCGA'

# GEO: sampleid -> sample_id, add dataset, and harmonize label names if needed
geo = geo.rename(columns={'sampleid': 'sample_id'})
geo['dataset'] = 'GEO'

# STAR: sample_barcode -> sample_id, dataset flag
star = star.rename(columns={'sample_barcode': 'sample_id'})
star['dataset'] = 'TCGA'

In [ ]:
import pandas as pd
import numpy as np

# If you're in a fresh runtime, re-read the 3 inputs first:
# tcga = pd.read_csv('tcga_sample_labelled (1).csv')
# geo  = pd.read_csv('geo_sample_labelled (1).csv')
# star = pd.read_csv('star_uuid_patient_map (1).csv')

# 1) Build correct short IDs from STAR aliquot barcodes
#    Example: TCGA-W5-AA33-01A -> TCGA-W5-AA33-01
barcode_col = 'sample_id'  # from star_uuid_patient_map-1-1-2.csv[file:1336]

def aliquot_to_sample(s):
    s = str(s)
    parts = s.split('-')
    if len(parts) >= 4:
        # keep first 3 parts + first 2 chars of the last part
        parts[3] = parts[3][:2]
        return '-'.join(parts[:4])
    return s

star['sample_id_short'] = star[barcode_col].apply(aliquot_to_sample)

# 2) Build TCGA base again (minimal columns)
tcga_base_cols = ['sample_id', 'label', 'dataset']
tcga_base_cols = [c for c in tcga_base_cols if c in tcga.columns]
tcga_base = tcga[tcga_base_cols].drop_duplicates()

# 3) Flag TCGA samples that have STAR counts
tcga_base['has_star_counts'] = tcga_base['sample_id'].isin(star['sample_id_short'])

# 4) GEO base (as before, no STAR counts)
geo = geo.rename(columns={'sampleid': 'sample_id'})  # if not already done
geo['dataset'] = 'GEO'
geo_base = geo[['sample_id', 'label', 'dataset']].drop_duplicates()
geo_base['has_star_counts'] = np.nan

# 5) Combine and save new master
sample_meta = pd.concat([tcga_base, geo_base], ignore_index=True)
print(sample_meta['has_star_counts'].value_counts(dropna=False))

sample_meta.to_csv('sample_meta_master_fixed.csv', index=False)

has_star_counts
NaN    304
0.0     31
1.0      5
Name: count, dtype: int64


In [ ]:
star_ids = star['sample_id_short'].unique().tolist()
sample_meta[sample_meta['sample_id'].isin(star_ids)][['sample_id', 'has_star_counts']]

,sample_id,has_star_counts
18,TCGA-W5-AA30-01,1.0
20,TCGA-W5-AA33-01,1.0
27,TCGA-YR-A95A-01,1.0
28,TCGA-ZD-A8I3-01,1.0
32,TCGA-ZH-A8Y5-01,1.0


In [ ]:
# --- Block 4: GEO base ---

geo_base = geo[['sample_id', 'label', 'dataset']].drop_duplicates()

# GEO currently has no patient/subtype/stage info; fill with NaN
import numpy as np

for col in ['patient_id', 'subtype', 'stage_group', 'has_star_counts']:
    geo_base[col] = np.nan

# Reorder columns to match TCGA base
geo_base = geo_base[tcga_base.columns]

In [ ]:
# --- Block 5: Combine TCGA + GEO ---

sample_meta = pd.concat([tcga_base, geo_base], ignore_index=True)

print("Total unique samples:", sample_meta['sample_id'].nunique())
print(sample_meta.groupby(['dataset', 'label'])['sample_id'].count())
print(sample_meta['has_star_counts'].value_counts())

Total unique samples: 340
dataset  label 
GEO      Tumour    304
TCGA     Tumour     36
Name: sample_id, dtype: int64
has_star_counts
0.0    31
1.0     5
Name: count, dtype: int64


In [ ]:
# --- Block 6: Save master file ---

sample_meta.to_csv('sample_meta_master.csv', index=False)

from google.colab import files
files.download('sample_meta_master.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>